# 2.1 Razonamiento

Desplegar un modelo razonador y parametrizar distintos grados de razonamiento (low, medium, high)

In [15]:
from openai import AzureOpenAI

endpoint = "https://at22-resource.services.ai.azure.com/" 
api_key = "Fbq49xJ5uDDfwZ1Oa8UpCOZUe3JAjiChTn7t2uemWTE3wHzLXzNBJQQJ99CDACfhMk5XJ3w3AAAAACOGhips"

client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=api_key,
    api_version="2024-12-01-preview"
)

deployment_name = "o4-mini" 

def test_reasoning(esfuerzo="medium"):
    print(f"🧠 Ejecutando razonamiento nivel: {esfuerzo}...")
    
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "user", "content": "¿Es posible que la Biblioteca de Alejandría no se quemara, sino que sus textos fueran trasladados sistemáticamente? Analiza las pruebas lógicas."}
            ],        
            reasoning_effort=esfuerzo 
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"❌ Error detectado: {e}"

print(test_reasoning("low"))
print("-" * 30)
print(test_reasoning("high"))

🧠 Ejecutando razonamiento nivel: low...
A lo largo de los siglos han surgido diversas hipótesis que cuestionan la idea tradicional de que la Gran Biblioteca de Alejandría fue destruida de forma violenta y definitiva por un solo incendió o invasión. Revisemos los principales puntos a favor y en contra de la idea de un “trasvase sistemático” de sus fondos: 


1. Fuentes antiguas y cronología contradictoria  
  • No existe un solo relato contemporáneo que describa la quema total de la biblioteca.  
  • Se mencionan al menos tres episodios en que Alejandría sufrió daños:  
    – 48 a. C.: Julio César incendia barcos en el puerto y parte del barrio contiguo.  
    – 270–275: saqueos durante la guerra de Aureliano contra la reina Zenobia de Palmira.  
    – 391: decreto de Teófilo de Alejandría que conmina a destruir templos paganos, entre ellos el Serapeo, donde se guardaban papiros.  
  • Cada suceso puede haber afectado colecciones distintas y dispersas, más que un único depósito central.

# 2.2 Function calling

Activar un motor de búsqueda web para probar llamadas a funciones (function calling) que recuperen información externa.

In [7]:
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key="5EaDUhSEXI32eFBKI9lq1dwF1ouXyPaWJWJuE9jygP0fN5PAZtqXJQQJ99CDAC5RqLJXJ3w3AAAAACOGBT0Z",
    azure_endpoint="https://01-aig.services.ai.azure.com/",
    api_version="2024-08-01-preview"
)

tools = [{
    "type": "function",
    "function": {
        "name": "web_search",
        "description": "Busca información actualizada en internet sobre deportes y eventos en 2026.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Término de búsqueda"}
            },
            "required": ["query"]
        }
    }
}]

def ejecutar_flujo_busqueda(pregunta):    
    messages = [{"role": "user", "content": pregunta}]
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls
    
    if tool_calls:
        print(f"📡 El modelo ha solicitado buscar en internet: {tool_calls[0].function.arguments}")
                
        messages.append(response_message)
        
        for tool_call in tool_calls:            
            busqueda_simulada = (
                "Resultados de búsqueda (Abril 2026): El Atlético de Madrid ha llegado a cuartos de final "
                "tras eliminar al Tottenham. Las casas de apuestas le dan un 20% de probabilidades "
                "de ganar la Champions este año, siendo el 'underdog' favorito."
            )
                        
            messages.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": "web_search",
                "content": busqueda_simulada
            })
                
        final_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )
        return final_response.choices[0].message.content
    
    return response_message.content

pregunta_usuario = "¿Cómo van las probabilidades del Atleti para ganar la Champions este 2026?"
respuesta = ejecutar_flujo_busqueda(pregunta_usuario)

print("\n--- RESPUESTA FINAL DEL MODELO ---")
print(respuesta)

📡 El modelo ha solicitado buscar en internet: {"query":"Atletico de Madrid Champions League 2026 odds"}

--- RESPUESTA FINAL DEL MODELO ---
En abril de 2026, el Atlético de Madrid ha llegado a los cuartos de final de la Champions League tras eliminar al Tottenham. Las casas de apuestas están evaluando sus probabilidades de ganar el torneo en un 20%, lo que lo clasifica como un "underdog" favorito. Esto indica que, aunque no son los principales favoritos, tienen una buena oportunidad de competir por el título.
